In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings("ignore")

# Loading the Dataset

In [ ]:
def load_dataset(filepath="cmu_dataset.csv"):
    """
    Load the CMU Keystroke dataset.
    Columns: subject, sessionIndex, rep, H.period, DD.period.t, ...
    Returns: features DataFrame and subject list.

    For BBMAS dataset, change the subject_col and feature_cols accordingly.
    """
    df = pd.read_csv(filepath)

    subject_col = "subject"
    # All numeric columns except identifiers are features
    exclude_cols = ["subject", "sessionIndex", "rep"]
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    return df, subject_col, feature_cols


# Building the binary dataset for one user

#### For each user in turns, we are labelling their own rows as genuine(1) and an equal number or rows for other users as impostor(0)

In [ ]:
def build_binary_dataset(df, subject_col, feature_cols, target_user, impostor_ratio=1.0):
    """
    For a target user:
      - All their samples → Genuine (1)
      - Sampled rows from all other users → Impostor (0)

    impostor_ratio: ratio of impostor samples to genuine samples (default 1:1 balanced)
    """
    genuine = df[df[subject_col] == target_user][feature_cols].copy()
    impostor_pool = df[df[subject_col] != target_user][feature_cols].copy()

    n_impostor = min(int(len(genuine) * impostor_ratio), len(impostor_pool))
    impostor = impostor_pool.sample(n=n_impostor, random_state=42)

    genuine["label"] = 1
    impostor["label"] = 0

    combined = pd.concat([genuine, impostor], ignore_index=True)
    X = combined[feature_cols].values
    y = combined["label"].values
    return X, y

# Computation of FAR, FRR and EER

In [ ]:
def compute_far_frr_eer(y_true, y_scores):
    """
    Sweep thresholds over probability scores to compute:
      FAR  = False Acceptance Rate = FP / (FP + TN)  [impostors accepted as genuine]
      FRR  = False Rejection Rate  = FN / (FN + TP)  [genuine users rejected]
      EER  = threshold where FAR ≈ FRR
    Returns: far_list, frr_list, eer, eer_threshold
    """
    thresholds = np.linspace(0, 1, 200)
    far_list, frr_list = [], []

    for thresh in thresholds:
        y_pred = (y_scores >= thresh).astype(int)

        tp = np.sum((y_pred == 1) & (y_true == 1))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        tn = np.sum((y_pred == 0) & (y_true == 0))

        far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        frr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        far_list.append(far)
        frr_list.append(frr)

    far_arr = np.array(far_list)
    frr_arr = np.array(frr_list)

    # EER: point where FAR and FRR curves cross
    diff = np.abs(far_arr - frr_arr)
    eer_idx = np.argmin(diff)
    eer = (far_arr[eer_idx] + frr_arr[eer_idx]) / 2
    eer_threshold = thresholds[eer_idx]

    return far_arr, frr_arr, thresholds, eer, eer_threshold

# Defining the set of classifiers to be used for classification

In [ ]:
def get_classifiers():
    return {
        "SVM (RBF)": SVC(kernel="rbf", probability=True, C=1.0, gamma="scale", random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
    }

# Training and evalauting one user-model pair

In [ ]:
def evaluate_user_model(X, y, model, test_size=0.3):
    """
    Split → Scale → Train → Predict scores → Compute metrics.
    Returns dict of metrics.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model.fit(X_train, y_train)

    # Probability of class=1 (genuine)
    y_scores = model.predict_proba(X_test)[:, 1]
    y_pred = (y_scores >= 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    far_arr, frr_arr, thresholds, eer, eer_thresh = compute_far_frr_eer(y_test, y_scores)

    # FAR and FRR at default threshold (0.5)
    idx_05 = np.argmin(np.abs(thresholds - 0.5))
    far_at_05 = far_arr[idx_05]
    frr_at_05 = frr_arr[idx_05]

    return {
        "accuracy": acc,
        "far": far_at_05,
        "frr": frr_at_05,
        "eer": eer,
        "eer_threshold": eer_thresh,
    }

# The MAIN PIPELINE starts here

In [ ]:
def run_pipeline(filepath="cmu_dataset.csv", max_users=10):
    """
    Full pipeline:
      - Load dataset
      - For each user × model: build binary dataset, train, evaluate
      - Aggregate and display results
    """
    print("=" * 65)
    print("   KEYSTROKE BINARY AUTHENTICATION — PIPELINE")
    print("=" * 65)

    # Load
    df, subject_col, feature_cols = load_dataset(filepath)
    users = df[subject_col].unique()[:max_users]
    classifiers = get_classifiers()

    print(f"\nDataset : {filepath}")
    print(f"Users   : {len(users)}  (of {df[subject_col].nunique()} total)")
    print(f"Features: {len(feature_cols)}")
    print(f"Models  : {list(classifiers.keys())}\n")

    # Per-model result accumulator
    model_results = {name: {"accuracy": [], "far": [], "frr": [], "eer": []}
                     for name in classifiers}

    # Per-user detailed results
    detailed_rows = []

    for uid, user in enumerate(users):
        print(f"  [{uid+1:02d}/{len(users)}] User: {user}")
        X, y = build_binary_dataset(df, subject_col, feature_cols, user)

        if len(np.unique(y)) < 2:
            print("       ⚠  Skipped: only one class present")
            continue

        for model_name, model in classifiers.items():
            import copy
            m = copy.deepcopy(model)
            metrics = evaluate_user_model(X, y, m)

            model_results[model_name]["accuracy"].append(metrics["accuracy"])
            model_results[model_name]["far"].append(metrics["far"])
            model_results[model_name]["frr"].append(metrics["frr"])
            model_results[model_name]["eer"].append(metrics["eer"])

            detailed_rows.append({
                "User": user,
                "Model": model_name,
                "Accuracy": round(metrics["accuracy"] * 100, 2),
                "FAR (%)": round(metrics["far"] * 100, 2),
                "FRR (%)": round(metrics["frr"] * 100, 2),
                "EER (%)": round(metrics["eer"] * 100, 2),
            })

    # ── Summary Table ──────────────────────────────────────────
    print("\n" + "=" * 65)
    print("   AVERAGED RESULTS ACROSS ALL USERS")
    print("=" * 65)

    summary_rows = []
    for model_name, vals in model_results.items():
        row = {
            "Model": model_name,
            "Avg Accuracy (%)": round(np.mean(vals["accuracy"]) * 100, 2),
            "Avg FAR (%)":      round(np.mean(vals["far"]) * 100, 2),
            "Avg FRR (%)":      round(np.mean(vals["frr"]) * 100, 2),
            "Avg EER (%)":      round(np.mean(vals["eer"]) * 100, 2),
        }
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows).sort_values("Avg EER (%)").reset_index(drop=True)
    print(summary_df.to_string(index=False))

    # ── Best Model ─────────────────────────────────────────────
    best = summary_df.iloc[0]
    print(f"\n✅ Best model for Continuous Authentication : {best['Model']}")
    print(f"   → Avg EER  : {best['Avg EER (%)']:.2f}%")
    print(f"   → Avg FAR  : {best['Avg FAR (%)']:.2f}%")
    print(f"   → Avg FRR  : {best['Avg FRR (%)']:.2f}%")
    print(f"   → Accuracy : {best['Avg Accuracy (%)']:.2f}%")

    # ── Save CSVs ─────────────────────────────────────────────
    detailed_df = pd.DataFrame(detailed_rows)
    detailed_df.to_csv("results_detailed.csv", index=False)
    summary_df.to_csv("results_summary.csv", index=False)
    print("\n📄 Saved: results_detailed.csv & results_summary.csv")

    return summary_df, detailed_df

# Execution

In [ ]:
if __name__ == "__main__":
    # ── Option A: Use the CMU dataset (default) ──────────────
    # Download from: https://www.cs.cmu.edu/~keystroke/
    # Place DSL-StrongPasswordData.csv in the same folder.
    summary, detailed = run_pipeline(
        filepath="cmu_dataset.csv",
        max_users=51 # Increase to process more users (max 51 in CMU dataset)
    )

   KEYSTROKE BINARY AUTHENTICATION — PIPELINE

Dataset : cmu_dataset.csv
Users   : 51  (of 51 total)
Features: 31
Models  : ['SVM (RBF)', 'Random Forest', 'KNN (k=5)', 'Logistic Regression', 'MLP Neural Net']

  [01/51] User: s002
  [02/51] User: s003
  [03/51] User: s004
  [04/51] User: s005
  [05/51] User: s007
  [06/51] User: s008
  [07/51] User: s010
  [08/51] User: s011
  [09/51] User: s012
  [10/51] User: s013
  [11/51] User: s015
  [12/51] User: s016
  [13/51] User: s017
  [14/51] User: s018
  [15/51] User: s019
  [16/51] User: s020
  [17/51] User: s021
  [18/51] User: s022
  [19/51] User: s024
  [20/51] User: s025
  [21/51] User: s026
  [22/51] User: s027
  [23/51] User: s028
  [24/51] User: s029
  [25/51] User: s030
  [26/51] User: s031
  [27/51] User: s032
  [28/51] User: s033
  [29/51] User: s034
  [30/51] User: s035
  [31/51] User: s036
  [32/51] User: s037
  [33/51] User: s038
  [34/51] User: s039
  [35/51] User: s040
  [36/51] User: s041
  [37/51] User: s042
  [38/51] Use